# Descriptor Visualization and Quality Audit

This notebook is for judging whether the generated descriptors are meaningful enough to use as conditioning signals before they are connected to the diffusion model.

It has two jobs:

- visually inspect raw sensor inputs next to generated descriptor maps
- read the descriptor quality audit and surface degenerate or suspicious outputs early


In [ ]:
from pathlib import Path
import csv
import json
from statistics import mean

try:
    from PIL import Image, ImageOps, ImageDraw
    PIL_AVAILABLE = True
except ImportError:
    Image = None
    ImageOps = None
    ImageDraw = None
    PIL_AVAILABLE = False

try:
    from IPython.display import display
except ImportError:
    display = None

print('PIL available:', PIL_AVAILABLE)

In [ ]:
def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / 'data' / 'raw' / 'MulSen_AD').exists() and (candidate / 'src' / 'notebooks').exists():
            return candidate
    raise RuntimeError(f'Could not locate project root from {start}')


PROJECT_ROOT = find_project_root()
DATA_ROOT = PROJECT_ROOT / 'data'
DESCRIPTOR_ROOT = DATA_ROOT / 'processed' / 'descriptors'
AUDIT_JSON = DATA_ROOT / 'processed' / 'reports' / 'descriptor_quality_summary.json'
AUDIT_CSV = DATA_ROOT / 'processed' / 'reports' / 'descriptor_quality.csv'
INDEX_CSV = DATA_ROOT / 'processed' / 'manifests' / 'dataset_index.csv'

print('Project root:', PROJECT_ROOT)
print('Descriptor root:', DESCRIPTOR_ROOT)
print('Audit JSON exists:', AUDIT_JSON.exists())
print('Index CSV exists:', INDEX_CSV.exists())

## Audit Summary

In [ ]:
if AUDIT_JSON.exists():
    audit_summary = json.loads(AUDIT_JSON.read_text(encoding='utf-8'))
    print('checked_manifests:', audit_summary['checked_manifests'])
    print('manifests_with_errors:', audit_summary['manifests_with_errors'])
    print('error_count:', audit_summary['error_count'])
    print('warning_count:', audit_summary['warning_count'])
    print('\nPer-group mean stats:')
    for group, stats in audit_summary.get('per_group_means', {}).items():
        print(group)
        for key, value in sorted(stats.items()):
            print('  ', key, '=', value)
else:
    print('Run python scripts/run_data_pipeline.py first to populate the quality report.')

In [ ]:
if AUDIT_CSV.exists():
    rows = list(csv.DictReader(AUDIT_CSV.open(newline='', encoding='utf-8')))
    print('Issue rows:', len(rows))
    for row in rows[:10]:
        print(row)
else:
    print('No CSV quality report found yet.')

## Available Descriptor Manifests

In [ ]:
manifest_paths = sorted(DESCRIPTOR_ROOT.rglob('manifest.json'))
print('Descriptor manifests found:', len(manifest_paths))
for path in manifest_paths[:12]:
    print(path.relative_to(DATA_ROOT))

In [ ]:
def project_path(path_like):
    path = Path(path_like)
    return path if path.is_absolute() else PROJECT_ROOT / path


def load_descriptor_manifest(path):
    return json.loads(project_path(path).read_text(encoding='utf-8'))


def infer_sample_root(manifest_path):
    relative = manifest_path.relative_to(DESCRIPTOR_ROOT)
    parts = relative.parts
    descriptor_type, category, split, defect_label, sample_id = parts[:5]
    return {
        'descriptor_type': descriptor_type,
        'category': category,
        'split': split,
        'defect_label': defect_label,
        'sample_id': sample_id,
    }


def build_sample_lookup(index_csv):
    lookup = {}
    if not index_csv.exists():
        return lookup
    for row in csv.DictReader(index_csv.open(newline='', encoding='utf-8')):
        key = (row['category'], row['split'], row['defect_label'], row['sample_id'])
        lookup[key] = row
    return lookup


sample_lookup = build_sample_lookup(INDEX_CSV)
print('Sample lookup rows:', len(sample_lookup))

## Visual Inspection Helper

In [ ]:
if not PIL_AVAILABLE:
    raise RuntimeError('Pillow is required for the visualization notebook.')


def prepare_tile(path, title, size=(320, 220)):
    image = Image.open(project_path(path)).convert('RGB')
    tile = ImageOps.pad(image, size, color=(255, 255, 255))
    canvas = Image.new('RGB', (size[0], size[1] + 28), 'white')
    canvas.paste(tile, (0, 28))
    draw = ImageDraw.Draw(canvas)
    draw.text((8, 8), title, fill='black')
    return canvas


def contact_sheet(tiles, columns=3, background='lightgray'):
    if not tiles:
        return None
    tile_width, tile_height = tiles[0].size
    rows = (len(tiles) + columns - 1) // columns
    sheet = Image.new('RGB', (tile_width * columns, tile_height * rows), background)
    for index, tile in enumerate(tiles):
        x = (index % columns) * tile_width
        y = (index // columns) * tile_height
        sheet.paste(tile, (x, y))
    return sheet

In [ ]:
preferred = [path for path in manifest_paths if '/crossmodal/' in str(path)]
selected_manifest = preferred[0] if preferred else (manifest_paths[0] if manifest_paths else None)
selected_manifest

In [ ]:
if selected_manifest is None:
    raise RuntimeError('No descriptor manifests found. Generate descriptors first.')

info = infer_sample_root(selected_manifest)
key = (info['category'], info['split'], info['defect_label'], info['sample_id'])
record = sample_lookup.get(key)
descriptor_manifest = load_descriptor_manifest(selected_manifest)

print('Selected manifest:', selected_manifest.relative_to(DATA_ROOT))
print('Sample key:', key)
print('Matched record exists:', record is not None)
if record:
    print('RGB path:', record['rgb_path'])
    print('IR path:', record['ir_path'])
    print('Pointcloud path:', record['pointcloud_path'])


In [ ]:
tiles = []
if record:
    tiles.append(prepare_tile(record['rgb_path'], 'RGB raw'))
    tiles.append(prepare_tile(record['ir_path'], 'Infrared raw'))

for artifact in descriptor_manifest:
    path = project_path(artifact['path'])
    if artifact['kind'] == 'spatial' and path.suffix.lower() == '.png':
        tiles.append(prepare_tile(path, artifact['name']))

sheet = contact_sheet(tiles, columns=3)
if display is not None and sheet is not None:
    display(sheet)
else:
    sheet

## Quick Descriptor Heuristics

A descriptor is promising if:

- it is not blank or constant across the image
- it has sensible spatial support rather than pure noise
- sparse maps such as hotspots or agreement maps are neither fully empty nor nearly full for most samples
- point-cloud projections cover a meaningful fraction of the target plane
- cross-modal agreement is not trivially zero across all samples


In [ ]:
if AUDIT_JSON.exists():
    audit_summary = json.loads(AUDIT_JSON.read_text(encoding='utf-8'))
    print('Interpretation guide:')
    means = audit_summary.get('per_group_means', {})
    for group, stats in means.items():
        print('\n', group)
        if 'std' in stats:
            print('  average spatial std:', stats['std'])
        if 'nonzero_ratio' in stats:
            print('  average nonzero ratio:', stats['nonzero_ratio'])
        if 'coverage_ratio' in stats:
            print('  average projection coverage:', stats['coverage_ratio'])
        if 'agreement_std' in stats:
            print('  average agreement std:', stats['agreement_std'])
else:
    print('Run the audit script first.')

## Recommended Next Check

After this notebook looks reasonable on the pilot bundles, expand generation to a few anomalous `test` samples for the same categories and compare:

- raw RGB vs descriptor maps
- raw infrared vs hotspot map
- point-cloud projection vs agreement map
- audit metrics for good vs anomalous samples

That comparison is the next sanity gate before the descriptors are connected to the diffusion model.